# 01 — Référentiel : législateurs & commissions

**Rôle :** construire la table d'identité (clé **BioGuideID**) avec parti, chambre, état/district, variantes de noms et appartenance aux commissions. C'est la pièce qui réconciliera les noms incohérents des PTR plus tard.

**Source :** `unitedstates/congress-legislators` (domaine public).

**Cellule 1 — Imports & chemins.** Ce notebook est autonome : il re-détecte la racine et les quelques dossiers dont il a besoin.

In [ ]:
import requests, json
from collections import defaultdict
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
REFERENCE = PROJECT_ROOT / "data" / "reference"; REFERENCE.mkdir(parents=True, exist_ok=True)
RAW_REF   = REFERENCE / "raw"; RAW_REF.mkdir(parents=True, exist_ok=True)
REPORTS   = PROJECT_ROOT / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
USER_AGENT = "Ramify-QIS research (contact: <ton-email>)"
print("Référentiel ->", REFERENCE)

**Cellule 2 — Petit téléchargeur JSON.** Réutilisable, avec cache disque (idempotent) et User-Agent honnête.

In [ ]:
BASE = "https://theunitedstates.io/congress-legislators/"

def fetch_json(name: str, force: bool = False):
    # Télécharge {name}.json depuis congress-legislators, avec cache disque.
    path = RAW_REF / f"{name}.json"
    if path.exists() and not force:
        return json.loads(path.read_text())
    r = requests.get(BASE + f"{name}.json", headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    data = r.json()
    path.write_text(json.dumps(data))
    return data

**Cellule 3 — Téléchargement du référentiel.** Législateurs actuels + historiques, liste des commissions, et appartenance aux commissions.

In [ ]:
legislators_current    = fetch_json("legislators-current")
legislators_historical = fetch_json("legislators-historical")
committees_current     = fetch_json("committees-current")
committee_membership   = fetch_json("committee-membership-current")

print("Actuels     :", len(legislators_current))
print("Historiques :", len(legislators_historical))
print("Commissions :", len(committees_current))

**Cellule 4 — Aplatissement d'un législateur.** On garde l'identité et le **dernier mandat connu** (parti, chambre, état, district).

In [ ]:
def flatten_legislator(person: dict) -> dict:
    ids   = person.get("id", {})
    name  = person.get("name", {})
    terms = person.get("terms", [])
    last  = terms[-1] if terms else {}
    chamber = {"rep": "house", "sen": "senate"}.get(last.get("type"))
    return {
        "bioguide_id":   ids.get("bioguide"),
        "last_name":     name.get("last"),
        "first_name":    name.get("first"),
        "official_full": name.get("official_full"),
        "nickname":      name.get("nickname"),
        "party":         last.get("party"),
        "chamber":       chamber,
        "state":         last.get("state"),
        "district":      last.get("district"),
    }

rows = [flatten_legislator(p) for p in legislators_current + legislators_historical]
people = (pd.DataFrame(rows)
          .dropna(subset=["bioguide_id"])
          .drop_duplicates("bioguide_id")
          .reset_index(drop=True))
print("Législateurs uniques :", len(people))
people.head(3)

**Cellule 5 — Variantes de nom.** Les PTR écrivent les noms de façon inconsistante (« Bill » vs « William »). On prépare toutes les formes connues + un nom canonique.

In [ ]:
def name_variants(r) -> list:
    forms = set()
    if isinstance(r.official_full, str):
        forms.add(r.official_full)
    if isinstance(r.first_name, str) and isinstance(r.last_name, str):
        forms.add(f"{r.first_name} {r.last_name}")
    if isinstance(r.nickname, str) and isinstance(r.last_name, str):
        forms.add(f"{r.nickname} {r.last_name}")
    return sorted(forms)

people["declarant_name"] = people["official_full"].fillna(
    (people["first_name"].fillna("") + " " + people["last_name"].fillna("")).str.strip())
people["name_variants"] = people.apply(name_variants, axis=1)
people[["bioguide_id", "declarant_name", "name_variants"]].head(3)

**Cellule 6 — Nom des commissions.** On indexe les commissions par identifiant pour pouvoir les nommer.

In [ ]:
committee_name = {c["thomas_id"]: c["name"] for c in committees_current}
list(committee_name.items())[:3]

**Cellule 7 — Membre → liste de commissions.** On rattache à chaque BioGuideID ses commissions.

In [ ]:
member_committees = defaultdict(set)
for thomas_id, members in committee_membership.items():
    cname = committee_name.get(thomas_id, thomas_id)
    for m in members:
        if m.get("bioguide"):
            member_committees[m["bioguide"]].add(cname)

print("Membres avec >=1 commission :", len(member_committees))

**Cellule 8 — Flag des commissions sensibles.** Finance/Banking, Armed Services (Defense) et Intelligence, par mots-clés (robuste aux variantes de noms).

> ⚠️ *Limite connue :* `committee-membership-current` ne couvre que le **Congrès actuel**. L'appartenance **historique** aux commissions (au moment d'un trade passé) est un point à compléter plus tard.

In [ ]:
KEY_PATTERNS = ("financial", "banking", "finance", "armed services", "intelligence")

def committees_key_flag(committees: set) -> bool:
    blob = " | ".join(committees).lower()
    return any(p in blob for p in KEY_PATTERNS)

people["committee_membership"] = people["bioguide_id"].map(
    lambda b: sorted(member_committees.get(b, set())))
people["committees_key_flag"] = people["committee_membership"].map(committees_key_flag)
int(people["committees_key_flag"].sum())

**Cellule 9 — Sauvegarde du référentiel.** Les colonnes-listes sont sérialisées en JSON dans le CSV.

In [ ]:
out = people.copy()
for col in ["name_variants", "committee_membership"]:
    out[col] = out[col].map(json.dumps)
ref_path = REFERENCE / "legislators.csv"
out.to_csv(ref_path, index=False)
print("Écrit :", ref_path, "-", len(out), "lignes")

**Cellule 10 — Mini-rapport de couverture.**

In [ ]:
n = len(people)
n_comm = int((people["committee_membership"].map(len) > 0).sum())
n_key  = int(people["committees_key_flag"].sum())
print(f"Législateurs        : {n}")
print(f"Avec >=1 commission : {n_comm} ({n_comm/n:.0%})  [Congrès actuel]")
print(f"Sur commission clé  : {n_key}")

Référentiel prêt ✅ — passez à **`02_house_index`**.